# 02 — Bronze → Silver (Databricks)

Reads all CSVs from ADLS Bronze + MongoDB enrichment,  
cleans and joins into a unified DataFrame, writes Delta to Silver.

## 1. Read CSVs from ADLS Bronze

In [ ]:
base = 'abfss://olistdata@olistdatastorage1105.dfs.core.windows.net/'

def read_bronze(name):
    return spark.read.csv(
        base + f'Bronze/{name}',
        header=True, inferSchema=True
    )

olist_customer_df        = read_bronze('olist_customers_dataset.csv')
olist_geolocation_df     = read_bronze('olist_geolocation_dataset.csv')
olist_order_items_df     = read_bronze('olist_order_items_dataset.csv')
olist_order_payments_df  = read_bronze('olist_order_payments_dataset.csv')
olist_order_reviews_df   = read_bronze('olist_order_reviews_dataset.csv')
olist_orders_df          = read_bronze('olist_orders_dataset.csv')
olist_products_df        = read_bronze('olist_products_dataset.csv')
olist_sellers_df         = read_bronze('olist_sellers_dataset.csv')

print('All Bronze CSVs loaded')


## 2. Read Product Category Translation from MongoDB

> Credentials via Databricks Secrets — `dbutils.secrets.get(scope, key)`

In [ ]:
%pip install pymongo


In [ ]:
import pandas as pd
from pymongo import MongoClient

# ── Use Databricks Secrets in production ───────────────────────────────
# MONGO_HOST = dbutils.secrets.get(scope='olist', key='mongo_host')
# MONGO_PASS = dbutils.secrets.get(scope='olist', key='mongo_pass')
# ── Or env vars for local testing ──────────────────────────────────────
import os
MONGO_HOST = os.getenv('MONGO_HOST')
MONGO_PORT = os.getenv('MONGO_PORT', '27017')
MONGO_DB   = os.getenv('MONGO_DB')
MONGO_USER = os.getenv('MONGO_USER')
MONGO_PASS = os.getenv('MONGO_PASS')

uri    = f'mongodb://{MONGO_USER}:{MONGO_PASS}@{MONGO_HOST}:{MONGO_PORT}/{MONGO_DB}'
client = MongoClient(uri)
db     = client[MONGO_DB]

collection = db['product_category_translation']
print(f'Documents in collection: {collection.count_documents({})}')

mongo_pd_df = pd.DataFrame(list(collection.find())).drop(columns=['_id'])
product_category_translation_df = spark.createDataFrame(mongo_pd_df)
client.close()
product_category_translation_df.show(5)


## 3. Clean All DataFrames

In [ ]:
from pyspark.sql.functions import col, to_date, datediff

def clean(df, name):
    print(f'Cleaning {name}: {df.count()} rows before')
    cleaned = df.dropDuplicates().na.drop('all')
    print(f'             {cleaned.count()} rows after')
    return cleaned

olist_orders_df                  = clean(olist_orders_df,                 'olist_orders')
olist_order_items_df             = clean(olist_order_items_df,            'olist_order_items')
olist_order_payments_df          = clean(olist_order_payments_df,         'olist_order_payments')
olist_order_reviews_df           = clean(olist_order_reviews_df,          'olist_order_reviews')
olist_products_df                = clean(olist_products_df,               'olist_products')
olist_sellers_df                 = clean(olist_sellers_df,                'olist_sellers')
product_category_translation_df = clean(product_category_translation_df, 'product_category_translation')


## 4. Cast Date Columns

In [ ]:
olist_orders_df = (
    olist_orders_df
    .withColumn('order_purchase_timestamp',      to_date(col('order_purchase_timestamp')))
    .withColumn('order_approved_at',             to_date(col('order_approved_at')))
    .withColumn('order_delivered_carrier_date',  to_date(col('order_delivered_carrier_date')))
    .withColumn('order_delivered_customer_date', to_date(col('order_delivered_customer_date')))
    .withColumn('order_estimated_delivery_date', to_date(col('order_estimated_delivery_date')))
)

olist_order_reviews_df = (
    olist_order_reviews_df
    .withColumn('review_creation_date',   to_date(col('review_creation_date')))
    .withColumn('review_answer_timestamp', to_date(col('review_answer_timestamp')))
)


## 5. Compute Delivery KPIs

In [ ]:
olist_orders_df = (
    olist_orders_df
    .withColumn('estimated_delivery_days',
                datediff(col('order_estimated_delivery_date'), col('order_purchase_timestamp')))
    .withColumn('actual_delivery_days',
                datediff(col('order_delivered_customer_date'), col('order_purchase_timestamp')))
    .withColumn('delivery_days',
                datediff(col('order_delivered_customer_date'), col('order_delivered_carrier_date')))
    .withColumn('delay', col('actual_delivery_days') > col('estimated_delivery_days'))
)

display(olist_orders_df)


## 6. Join All Datasets

In [ ]:
# Chain joins — drop duplicate key columns as we go to avoid ambiguity
final_df = (
    olist_orders_df
    .join(olist_customer_df,       'customer_id', 'left')
    .join(olist_order_payments_df, 'order_id',    'left')
    .join(olist_order_items_df,    'order_id',    'left')
    .join(olist_products_df,       'product_id',  'left')
    .join(olist_sellers_df,        'seller_id',   'left')
    .join(product_category_translation_df, 'product_category_name', 'left')
)

display(final_df)
print(f'Final schema: {len(final_df.columns)} columns, {final_df.count()} rows')


## 7. Write Silver Layer (Delta)

In [ ]:
silver_path = 'abfss://olistdata@olistdatastorage1105.dfs.core.windows.net/Silver/full_orders/'

final_df.write \
    .format('delta') \
    .mode('overwrite') \
    .save(silver_path)

print(f'Silver layer written to {silver_path}')
